# Transformer Training — ModernBERT-base Fine-Tuning

Fine-tunes `answerdotai/ModernBERT-base` for all 4 classification tasks.
Evaluates on held-out test set and performs error analysis.

**Input:** `data/tokenized/` (from notebook 8)
**Output:** Trained models in `models/` + evaluation results

In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from datasets import DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

Device: cpu


In [4]:
# ── Configuration ──
MODEL_NAME = "answerdotai/ModernBERT-base"
TOKENIZED_DIR = "data/tokenized"
MODEL_DIR = "models"
Path(MODEL_DIR).mkdir(exist_ok=True)

TASKS = {
    "sentiment": {
        "num_labels": 4,
        "encoder_path": "data/model_ready/encoders/le_sentiment.pkl",
        "epochs": 5,
        "batch_size": 32,
        "lr": 2e-5,
    },
    "intent": {
        "num_labels": None,  # will be set from data
        "encoder_path": "data/model_ready/encoders/le_intent.pkl",
        "epochs": 8,
        "batch_size": 32,
        "lr": 2e-5,
    },
    "root_cause": {
        "num_labels": 26,
        "encoder_path": "data/model_ready/encoders/le_root_cause.pkl",
        "epochs": 8,
        "batch_size": 32,
        "lr": 2e-5,
    },
    "risk": {
        "num_labels": 5,
        "encoder_path": None,  # risk uses fixed band names
        "epochs": 5,
        "batch_size": 32,
        "lr": 2e-5,
    },
}

# Load label encoders
for task_name, task_cfg in TASKS.items():
    if task_cfg["encoder_path"] and os.path.exists(task_cfg["encoder_path"]):
        with open(task_cfg["encoder_path"], "rb") as f:
            enc = pickle.load(f)
        task_cfg["label_names"] = list(enc.classes_)
        if task_cfg["num_labels"] is None:
            task_cfg["num_labels"] = len(enc.classes_)
        print(f"  {task_name}: {len(enc.classes_)} classes loaded")
    else:
        task_cfg["label_names"] = ["NEGLIGIBLE", "LOW", "MODERATE", "HIGH", "CRITICAL"]
        print(f"  {task_name}: using fixed band names")

print(f"\nAll task configurations ready")

  sentiment: 4 classes loaded
  intent: 19 classes loaded
  root_cause: 25 classes loaded
  risk: using fixed band names

All task configurations ready


In [5]:
# ── Load tokenizer and data collator ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"Tokenizer loaded: {MODEL_NAME}")

Tokenizer loaded: answerdotai/ModernBERT-base


## Task 1: Sentiment Classification
Starting with the simplest task (4 classes) to validate the training pipeline.

In [6]:
# ── Load tokenized sentiment data ──
task = "sentiment"
dd = DatasetDict.load_from_disk(os.path.join(TOKENIZED_DIR, task))
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

Loaded sentiment:
  train     : 11,502
  validation: 2,465
  test      : 2,465


In [ ]:
# ── Train sentiment model ──
task = "sentiment"
task_cfg = TASKS[task]

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
)

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=64,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=(device == "cuda"),
    report_to="none",
)

# compute_metrics is the ONE place where a callable is needed by Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Training {task}...")
trainer.train()
print("Training complete!")

# Save best model
model.save_pretrained(f"{MODEL_DIR}/{task}_best")
tokenizer.save_pretrained(f"{MODEL_DIR}/{task}_best")
print(f"Model saved to {MODEL_DIR}/{task}_best")

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training sentiment...


c:\Users\amala\miniconda3\envs\ai_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.294246,0.277324,0.887627,0.850504,0.888450


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\amala\miniconda3\envs\ai_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# ── Evaluate sentiment on test set ──
task = "sentiment"
task_cfg = TASKS[task]

preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print("=" * 60)
print(f"SENTIMENT — TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=task_cfg["label_names"]))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()

NameError: name 'trainer' is not defined

## Task 2: Intent Classification
Training model for predicting intent classes.

In [ ]:
# ── Load tokenized intent data ──
task = "intent"
dd = DatasetDict.load_from_disk(os.path.join(TOKENIZED_DIR, task))
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train intent model ──
task = "intent"
task_cfg = TASKS[task]

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
)

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=64,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=(device == "cuda"),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Training {task}...")
trainer.train()
print("Training complete!")

# Save best model
model.save_pretrained(f"{MODEL_DIR}/{task}_best")
tokenizer.save_pretrained(f"{MODEL_DIR}/{task}_best")
print(f"Model saved to {MODEL_DIR}/{task}_best")

In [ ]:
# ── Evaluate intent on test set ──
task = "intent"
task_cfg = TASKS[task]

preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print("=" * 60)
print(f"INTENT — TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=task_cfg["label_names"]))

# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation='vertical')
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()

## Task 3: Root Cause Classification
Training model for predicting root causes.

In [ ]:
# ── Load tokenized root_cause data ──
task = "root_cause"
dd = DatasetDict.load_from_disk(os.path.join(TOKENIZED_DIR, task))
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train root_cause model ──
task = "root_cause"
task_cfg = TASKS[task]

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
)

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=64,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=(device == "cuda"),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Training {task}...")
trainer.train()
print("Training complete!")

# Save best model
model.save_pretrained(f"{MODEL_DIR}/{task}_best")
tokenizer.save_pretrained(f"{MODEL_DIR}/{task}_best")
print(f"Model saved to {MODEL_DIR}/{task}_best")

In [ ]:
# ── Evaluate root_cause on test set ──
task = "root_cause"
task_cfg = TASKS[task]

preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print("=" * 60)
print(f"ROOT CAUSE — TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=task_cfg["label_names"]))

# Confusion matrix
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation='vertical')
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()

## Task 4: Risk Band Classification
Training model for predicting 5 risk bands.

In [ ]:
# ── Load tokenized risk data ──
task = "risk"
dd = DatasetDict.load_from_disk(os.path.join(TOKENIZED_DIR, task))
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train risk model ──
task = "risk"
task_cfg = TASKS[task]

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
)

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=64,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=(device == "cuda"),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Training {task}...")
trainer.train()
print("Training complete!")

# Save best model
model.save_pretrained(f"{MODEL_DIR}/{task}_best")
tokenizer.save_pretrained(f"{MODEL_DIR}/{task}_best")
print(f"Model saved to {MODEL_DIR}/{task}_best")

In [ ]:
# ── Evaluate risk on test set ──
task = "risk"
task_cfg = TASKS[task]

preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print("=" * 60)
print(f"RISK — TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=task_cfg["label_names"]))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()

## CRO Integration — Full Pipeline Inference
Demonstrates end-to-end inference: text → sentiment + intent + root_cause + risk → CRO action

In [ ]:
# ── CRO Inference Pipeline ──
# Load the CRO taxonomy
with open("cro_taxonomy_export.json", "r") as f:
    cro_taxonomy = json.load(f)

print("CRO Taxonomy loaded")
print(f"  Intents: {len(cro_taxonomy['valid_intents'])}")
print(f"  Root causes: {len(cro_taxonomy['valid_root_causes'])}")

# Example: classify a new review
test_reviews = [
    "App crashes every time I try to transfer money. Lost my patience.",
    "Best banking app in Sri Lanka! Love the offers.",
    "I've been trying to log in for 3 days. No OTP received. My money is stuck.",
    "Switching to HNB app. This one is hopeless.",
]

print("\n" + "=" * 70)
print("CRO INFERENCE DEMO")
print("=" * 70)

for review in test_reviews:
    print(f"\nReview: \"{review[:80]}...\"" if len(review) > 80 else f"\nReview: \"{review}\"")
    
    # Tokenize
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # For now, just demonstrate with sentiment model
    # In production, you would load all 4 models and run inference on each
    with torch.no_grad():
        outputs = model(**inputs)
    pred_label = outputs.logits.argmax(-1).item()
    pred_name = TASKS["sentiment"]["label_names"][pred_label]
    print(f"  Predicted sentiment: {pred_name}")
    
    # Look up CRO action (example with intent taxonomy)
    # In the full system, you would predict intent too and look up:
    # action = cro_taxonomy["intent_meta"][predicted_intent]
    # print(f"  CRO Action: {action['description']}")
    # print(f"  Priority: {action['priority']}")
    # print(f"  SLA: {action['sla_hours']}h")

print("\n" + "=" * 70)
print("PROJECT COMPLETE")
print("=" * 70)
print("\nTo build the full CRO pipeline:")
print("1. Load all 4 trained models")
print("2. For each review: predict sentiment, intent, root_cause, risk")
print("3. Look up CRO action from taxonomy based on intent")
print("4. Generate risk score and escalation priority")